In [41]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

In [42]:
load_dotenv(override=True)  # load .env first; override any stale kernel env vars
MODEL = os.getenv("MODEL", "openai/gpt-oss-20b")

llm = ChatGroq(
  api_key=os.getenv("GROQ_API_KEY"),
  model=MODEL
)

def ask(system_prompt: str, user_prompt: str):
  if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY environment variable is not set.")
  return llm.invoke(
    [
      SystemMessage(content=system_prompt),
      HumanMessage(content=user_prompt)
    ]
  ).content

## State


In [43]:
class TicketState(TypedDict):
  text: str
  normalized_text: str
  category: str

## Nodes


In [44]:
def normalised(s: TicketState) -> dict:
  return {"normalized_text": " ".join(s["text"].lower().split())}

In [45]:
def classify(s: TicketState) -> dict:
  category = ask(
    "Classify as billing, technical, support or other. Output one lowercase word only.", 
    s["normalized_text"].strip().lower()
  )

  return {"category": category if category in ["billing", "technical", "support", "other"] else "other"}

## Create the graph

In [46]:
builder= StateGraph(TicketState)
builder.add_node("normalized", normalised)
builder.add_node("classify", classify)

builder.add_edge(START, "normalized")
builder.add_edge("normalized", "classify")
builder.add_edge("classify", END)

## Compile the Graph

In [47]:
graph = builder.compile()

graph.invoke({"text": "I need help with my bill, I think I was overcharged."})

{'text': 'I need help with my bill, I think I was overcharged.',
 'normalized_text': 'i need help with my bill, i think i was overcharged.',
 'category': 'billing'}